# 01 — Data Preparation: Cleaning & Merge
**Group P33** — Mika Ohayon, Akshaya Somasundaram, Harini Champooranan

| Notebook | Contents |
|---|---|
| `01_Data_Prep_Cleaning_Merge.ipynb` | Clean the raw BLS 2025, BLS 2022, and AIOE files, then merge them into `finalized_data.csv` |
| `02_Exploratory_Statistics.ipynb` | Descriptive statistics, correlations, model-form checks, and first look at 2022 → 2025 change |
| `03_Findings_Visuals_Models.ipynb` | Interactive visuals 1–10, machine learning models, results, and conclusions |

**Run them in order (01 → 02 → 03).** Notebooks 02 and 03 load `finalized_data.csv`, which notebook 01 creates.

### This notebook
1. Load the three raw files from `data/`
2. Clean both BLS years with the same rules
3. Check the AIOE file
4. Merge everything into one occupation-level table → **`finalized_data.csv`**

**Datasets**
| # | Source | File | Role |
|---|---|---|---|
| 1 | BLS OEWS, May 2025, national cross-industry | `data/national_M2025_dl.xlsx` | Wages and employment now |
| 2 | Felten, Raj & Seamans AIOE, Appendix A | `data/AIOE_DataAppendix.xlsx` | AI exposure score per occupation |
| 3 | BLS OEWS, May 2022, national cross-industry | `data/national_M2022_dl.xlsx` | Wages and employment before ChatGPT (Nov 2022) |


In [1]:
# If a package is missing, uncomment and run once:
# %pip install pandas numpy openpyxl

import os
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 140)

DATA_DIR = 'data'
for f in ['national_M2025_dl.xlsx', 'national_M2022_dl.xlsx', 'AIOE_DataAppendix.xlsx']:
    path = os.path.join(DATA_DIR, f)
    print(('found   ' if os.path.exists(path) else 'MISSING ') + path)

found   data/national_M2025_dl.xlsx
found   data/national_M2022_dl.xlsx
found   data/AIOE_DataAppendix.xlsx


## Part 1 — Data Cleaning
We apply **exactly the same rules** to the 2025 and 2022 BLS files so the two years line up. These are the rules from our Analysis phase:

| Issue | What we do | Why |
|---|---|---|
| `JOBS_1000`, `LOC_QUOTIENT`, `PCT_TOTAL`, `PCT_RPT` are 100% blank | Drop the columns | Only filled in state/metro/industry files, not the national file |
| `ANNUAL` / `HOURLY` blank | Recode to `False` | Blank means "flag doesn't apply", not unknown |
| Wage `*` | → `NaN`, **not imputed** | Occupation isn't reported on that pay basis |
| Wage `#` | → `NaN`, **not imputed** | Wage is above the BLS top-code (≈ $239,200/yr); exact value unpublished |
| Employment `**` | → `NaN` | Estimate not released |
| Duplicate `OCC_CODE` (broad + detailed rows) | Keep both | Two legitimate estimates at different levels; we use only `detailed` rows later |
| Rows deleted | None | Nothing was a data-entry error |

In [2]:
EMPTY_COLS = ['JOBS_1000', 'LOC_QUOTIENT', 'PCT_TOTAL', 'PCT_RPT']
WAGE_COLS = ['H_MEAN', 'H_PCT10', 'H_PCT25', 'H_MEDIAN', 'H_PCT75', 'H_PCT90',
             'A_MEAN', 'A_PCT10', 'A_PCT25', 'A_MEDIAN', 'A_PCT75', 'A_PCT90']


def clean_oews(path, year):
    # Clean one BLS OEWS national file with the project's wrangling rules.
    raw = pd.read_excel(path, sheet_name=0)
    print(f'===== OEWS {year} =====')
    print('Raw shape:', raw.shape)

    df_ = raw.drop(columns=[c for c in EMPTY_COLS if c in raw.columns]).copy()

    for c in ['ANNUAL', 'HOURLY']:
        df_[c] = df_[c].fillna(False).astype(bool)

    # count BLS special codes before converting them to NaN
    codes = {}
    for c in WAGE_COLS + ['TOT_EMP']:
        s = df_[c].astype(str).str.strip()
        codes[c] = {'*': int((s == '*').sum()), '#': int((s == '#').sum()), '**': int((s == '**').sum())}
        df_[c] = pd.to_numeric(df_[c], errors='coerce')
    code_tbl = pd.DataFrame(codes).T
    print('BLS codes converted to NaN (not imputed):')
    print(code_tbl[code_tbl.sum(axis=1) > 0].to_string())

    # range / format / duplicate checks
    print('\nChecks (all should be 0):')
    print('  negative TOT_EMP:              ', int((df_['TOT_EMP'] < 0).sum()))
    print('  negative wages:                ', int((df_[WAGE_COLS] < 0).sum().sum()))
    print('  EMP_PRSE/MEAN_PRSE outside 0-100:',
          int((~df_[['EMP_PRSE', 'MEAN_PRSE']].apply(lambda s: s.between(0, 100))).sum().sum()))
    print('  OCC_CODE not DD-DDDD:          ', int((~df_['OCC_CODE'].str.match(r'^\d{2}-\d{4}$')).sum()))
    print('  TOT_EMP not rounded to 10:     ', int((df_['TOT_EMP'].dropna() % 10 != 0).sum()))
    print('  fully duplicated rows:         ', int(df_.duplicated().sum()))
    dup = df_[df_.duplicated(subset=['OCC_CODE'], keep=False)]
    print(f'  repeated OCC_CODE (broad+detailed pairs, kept): {dup["OCC_CODE"].nunique()}')
    print('Cleaned shape:', df_.shape, '\n')
    return df_


oews25 = clean_oews(os.path.join(DATA_DIR, 'national_M2025_dl.xlsx'), 2025)
oews22 = clean_oews(os.path.join(DATA_DIR, 'national_M2022_dl.xlsx'), 2022)

===== OEWS 2025 =====
Raw shape: (1401, 32)
BLS codes converted to NaN (not imputed):
           *  #  **
H_MEAN    81  0   0
H_PCT10   81  0   0
H_PCT25   81  0   0
H_MEDIAN  81  0   0
H_PCT75   81  0   0
H_PCT90   81  0   0
A_MEAN     7  0   0
A_PCT10    7  0   0
A_PCT25    7  0   0
A_MEDIAN   7  0   0
A_PCT75    7  0   0
A_PCT90    7  0   0

Checks (all should be 0):
  negative TOT_EMP:               0
  negative wages:                 0
  EMP_PRSE/MEAN_PRSE outside 0-100: 0
  OCC_CODE not DD-DDDD:           0
  TOT_EMP not rounded to 10:      0
  fully duplicated rows:          0
  repeated OCC_CODE (broad+detailed pairs, kept): 7
Cleaned shape: (1401, 28) 



===== OEWS 2022 =====
Raw shape: (1402, 32)
BLS codes converted to NaN (not imputed):
           *   #  **
H_MEAN    83   0   0
H_PCT10   83   0   0
H_PCT25   83   1   0
H_MEDIAN  83  12   0
H_PCT75   83  24   0
H_PCT90   83  51   0
A_MEAN     7   0   0
A_PCT10    7   0   0
A_PCT25    7   1   0
A_MEDIAN   7  12   0
A_PCT75    7  26   0
A_PCT90    7  57   0

Checks (all should be 0):
  negative TOT_EMP:               0
  negative wages:                 0
  EMP_PRSE/MEAN_PRSE outside 0-100: 0
  OCC_CODE not DD-DDDD:           0
  TOT_EMP not rounded to 10:      0
  fully duplicated rows:          0
  repeated OCC_CODE (broad+detailed pairs, kept): 7
Cleaned shape: (1402, 28) 



In [3]:
# AIOE Appendix A — checked, no cleaning needed
aioe = pd.read_excel(os.path.join(DATA_DIR, 'AIOE_DataAppendix.xlsx'), sheet_name='Appendix A')
print('AIOE shape:', aioe.shape)
print('Missing values:', int(aioe.isna().sum().sum()))
print('Duplicated rows / SOC codes:', int(aioe.duplicated().sum()), '/', int(aioe['SOC Code'].duplicated().sum()))
print('SOC Code not DD-DDDD:', int((~aioe['SOC Code'].str.match(r'^\d{2}-\d{4}$')).sum()))
print('Implausible AIOE (|z| > 4):', int((aioe['AIOE'].abs() > 4).sum()))
print(f"AIOE mean = {aioe['AIOE'].mean():.3f}, std = {aioe['AIOE'].std():.3f}  (standardized, as expected)")

# Save cleaned files (same names as before, plus the new 2022 file)
oews25.to_csv('dataset1_new.csv', index=False)
aioe.to_csv('dataset2_new.csv', index=False)
oews22.to_csv('dataset3_2022_new.csv', index=False)
print('\nSaved dataset1_new.csv', oews25.shape, '| dataset2_new.csv', aioe.shape, '| dataset3_2022_new.csv', oews22.shape)

AIOE shape: (774, 3)
Missing values: 0
Duplicated rows / SOC codes: 0 / 0
SOC Code not DD-DDDD: 0
Implausible AIOE (|z| > 4): 0
AIOE mean = -0.000, std = 1.000  (standardized, as expected)

Saved dataset1_new.csv (1401, 28) | dataset2_new.csv (774, 3) | dataset3_2022_new.csv (1402, 28)


> **Key takeaways (cleaning)**
>
> **Cleaning summary.** Both BLS years passed every range, format, and duplicate check, so no rows were deleted. The only missing values are BLS's own codes: `*` means the job isn't reported on that pay basis, and `#` means the wage is above the top-code, which happens in 2022 for a few top-paid jobs like surgeons. We leave both missing instead of inventing numbers. The AIOE file needed no cleaning.

---

## Part 2 — Merge → `finalized_data.csv`
1. Keep only **detailed** occupations. This removes the "All Occupations" total and the group subtotals, and resolves the 7 broad/detailed duplicate codes.
2. Join 2025 BLS to AIOE on SOC code. 673 of 830 occupations match. The rest are SOC codes that changed after AIOE was published, since AIOE uses the 2010 codes and BLS uses the 2018 codes.
3. Drop occupations with no 2025 median wage, which leaves 670.
4. Join the 2022 values on the same occupation code. Both years use the 2018 SOC codes, so almost every occupation matches.
5. Add the features the visuals and models use, then save everything as **`finalized_data.csv`**.

In [4]:
MAJOR_NAMES = {
    '11': 'Management', '13': 'Business & Financial', '15': 'Computer & Mathematical',
    '17': 'Architecture & Engineering', '19': 'Life, Physical & Social Science',
    '21': 'Community & Social Service', '23': 'Legal', '25': 'Education & Library',
    '27': 'Arts, Design & Media', '29': 'Healthcare Practitioners',
    '31': 'Healthcare Support', '33': 'Protective Service', '35': 'Food Prep & Serving',
    '37': 'Building & Grounds Cleaning', '39': 'Personal Care & Service', '41': 'Sales',
    '43': 'Office & Admin Support', '45': 'Farming, Fishing & Forestry',
    '47': 'Construction & Extraction', '49': 'Installation & Repair',
    '51': 'Production', '53': 'Transportation & Moving', '55': 'Military',
}
HI, LO = 'Higher-skill (SOC 11–29)', 'Lower-skill (SOC 31–53)'

det25 = oews25[oews25['O_GROUP'].str.lower() == 'detailed'].copy()
check = det25.merge(aioe, left_on='OCC_CODE', right_on='SOC Code', how='outer', indicator=True)
print('2025 BLS ↔ AIOE match:\n', check['_merge'].value_counts().to_string())

df = check[check['_merge'] == 'both'].drop(columns='_merge').copy()
df = df.dropna(subset=['A_MEDIAN', 'AIOE', 'TOT_EMP'])
df = df[df['TOT_EMP'] > 0]

# features (same definitions as the Design phase)
df['soc_major'] = df['OCC_CODE'].str[:2]
df['major_group'] = df['soc_major'].map(MAJOR_NAMES).fillna(df['soc_major'])
df['high_skill'] = (df['soc_major'].astype(int) <= 29).astype(int)
df['skill_tier'] = np.where(df['high_skill'] == 1, HI, LO)
df['AIOE_x_skill'] = df['AIOE'] * df['high_skill']
df['log_wage'] = np.log(df['A_MEDIAN'])
df['log_emp'] = np.log10(df['TOT_EMP'])

BAND_LABELS = ['Bottom 20%', '20–40%', '40–60%', '60–80%', 'Top 20%']
BAND_ORDER = BAND_LABELS[::-1]
df['wage_band'] = pd.qcut(df['A_MEDIAN'].rank(method='first'), 5, labels=BAND_LABELS).astype(str)
df['aioe_bin_start'] = np.floor(df['AIOE'] / 0.5) * 0.5
df['aioe_bin'] = df['aioe_bin_start'].map(lambda s: f'{s:+.1f} to {s + 0.5:+.1f}')
BIN_ORDER = [f'{s:+.1f} to {s + 0.5:+.1f}' for s in sorted(df['aioe_bin_start'].unique())]

# add 2022 values
det22 = (oews22[oews22['O_GROUP'].str.lower() == 'detailed'][['OCC_CODE', 'TOT_EMP', 'A_MEDIAN']]
         .rename(columns={'TOT_EMP': 'TOT_EMP_22', 'A_MEDIAN': 'A_MEDIAN_22'}))
df = df.merge(det22, on='OCC_CODE', how='left')
print(f"\n2025 occupations with a 2022 match: {df['TOT_EMP_22'].notna().sum()} of {len(df)}")
print(f"  of those, missing 2022 median wage (top-coded #): {(df['TOT_EMP_22'].notna() & df['A_MEDIAN_22'].isna()).sum()}")

df = df.rename(columns={'OCC_TITLE': 'occupation'})
df = df[['OCC_CODE', 'occupation', 'soc_major', 'major_group', 'skill_tier', 'high_skill',
         'AIOE', 'AIOE_x_skill', 'TOT_EMP', 'log_emp', 'A_MEDIAN', 'log_wage',
         'A_PCT10', 'A_PCT25', 'A_PCT75', 'A_PCT90',
         'wage_band', 'aioe_bin_start', 'aioe_bin', 'TOT_EMP_22', 'A_MEDIAN_22']].reset_index(drop=True)

df.to_csv('finalized_data.csv', index=False)
print('\nSaved finalized_data.csv', df.shape)
df.head()

2025 BLS ↔ AIOE match:
 _merge
both          673
left_only     157
right_only    101

2025 occupations with a 2022 match: 669 of 670
  of those, missing 2022 median wage (top-coded #): 1

Saved finalized_data.csv (670, 21)


,OCC_CODE,occupation,soc_major,major_group,skill_tier,high_skill,AIOE,AIOE_x_skill,TOT_EMP,log_emp,A_MEDIAN,log_wage,A_PCT10,A_PCT25,A_PCT75,A_PCT90,wage_band,aioe_bin_start,aioe_bin,TOT_EMP_22,A_MEDIAN_22
0,11-1011,Chief Executives,11,Management,Higher-skill (SOC 11–29),1,1.334246,1.334246,204350.0,5.310375,213990.0,12.273685,75700.0,129540.0,356200.0,507730.0,Top 20%,1.0,+1.0 to +1.5,199240.0,189520.0
1,11-1021,General and Operations Managers,11,Management,Higher-skill (SOC 11–29),1,0.574877,0.574877,3503020.0,6.544443,105770.0,11.569022,50090.0,72320.0,167280.0,253390.0,Top 20%,0.5,+0.5 to +1.0,3376680.0,98100.0
2,11-2011,Advertising and Promotions Managers,11,Management,Higher-skill (SOC 11–29),1,1.294387,1.294387,21470.0,4.331832,133660.0,11.803055,63300.0,91370.0,201050.0,286240.0,Top 20%,1.0,+1.0 to +1.5,22010.0,127830.0
3,11-2021,Marketing Managers,11,Management,Higher-skill (SOC 11–29),1,1.315032,1.315032,395240.0,5.596861,166790.0,12.024491,90260.0,123020.0,216410.0,293610.0,Top 20%,1.0,+1.0 to +1.5,328570.0,140040.0
4,11-2022,Sales Managers,11,Management,Higher-skill (SOC 11–29),1,1.266280,1.266280,637080.0,5.804194,148270.0,11.906790,73170.0,100360.0,207340.0,290540.0,Top 20%,1.0,+1.0 to +1.5,536390.0,130600.0


> **Key takeaways**
>
> - **670 occupations** have an AIOE score and a 2025 median wage. These are the occupations every later analysis uses.
> - **669 of the 670** also appear in the 2022 file, since both BLS years use the same 2018 SOC codes. Only 1 is missing a 2022 wage because it was top-coded (`#`).
> - 157 BLS occupations have no AIOE score, because AIOE uses the older 2010 SOC codes. This is a known coverage gap, not a cleaning error.
> - `finalized_data.csv` has one row per occupation: 2025 wages and employment, 2022 wages and employment, AI exposure, skill tier, and the features the models use.

---
### Output files
| File | Contents |
|---|---|
| `dataset1_new.csv` | Cleaned BLS 2025 (all 1,401 rows) |
| `dataset2_new.csv` | AIOE Appendix A (774 rows) |
| `dataset3_2022_new.csv` | Cleaned BLS 2022 (all 1,402 rows) |
| **`finalized_data.csv`** | **Merged analysis table (670 occupations) used by notebooks 02 and 03** |

**Next:** open `02_Exploratory_Statistics.ipynb`.